In [ ]:
# ---- configuration -------------------------------------------------------------------------
QUICK = False            # True: reduced seeds for a fast check; False: full protocol of the paper
RUN_PART_A = False        # EM harness: Sections III-A, III-E, III-F, III-H
RUN_PART_B = False        # neural architecture: Sections II-F, II-G, eighth/ninth limitations
RUN_REAL_NOTES = False    # Part B9: 600 public chest-radiograph + case-note pairs (downloaded from GitHub)
OUT_DIR = "paper_outputs"            # all CSVs, JSONs and figures go here
REAL_DATA_DIR = "real_case_notes"    # where Part B9 stores the downloaded public data
# Part C (Section III-D public-data checks). These datasets cannot be redistributed: download them
# yourself and point the paths below at them; leave None to skip.
IU_DATA_DIR = "/home/user/Jayashree/notebook_run/openi_data"       # e.g. "/data/chest-xrays-indiana-university" (indiana_reports.csv, indiana_projections.csv, images)
                         # Kaggle: https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university
NIH_LABELS_DIR = None    # e.g. "/data/four_findings_expert_labels" (individual_readers.csv, validation/test_labels.csv)
NIH_IMAGES_DIR = None    # e.g. "/data/nih-chest-xrays" (ChestX-ray14 PNGs, any sub-folder layout)
                         # Kaggle (uploaded by the NIH): https://www.kaggle.com/datasets/nih-chest-xrays/data
# Part D (the datasets of Table VII). PhysioNet credentialed access (MIMIC-CXR, MIMIC-CXR-JPG, RadGraph,
# VinDr-CXR, REFLACX, CorBenchX) or Stanford AIMI registration (CheXpert), in your own name. Download them
# yourself and point the paths below at the unzipped folders; leave None to skip. Do not use unofficial
# re-uploads of credentialed data: the data use agreement forbids redistribution.
RUN_PART_D = True
RADGRAPH_DIR = None        # physionet.org/content/radgraph/1.0.0  (train/dev/test.json, MIMIC-CXR_graphs.json)
MIMIC_JPG_DIR = None       # physionet.org/content/mimic-cxr-jpg/2.1.0  (metadata, chexpert csv, files/pXX/...jpg)
MIMIC_REPORTS_DIR = None   # physionet.org/content/mimic-cxr/2.1.0  (unzipped mimic-cxr-reports.zip: files/pXX/pXXXXXXXX/sXXXXXXXX.txt)
MIMIC_PROVIDER_CSV = None  # cxr-provider-list.csv.gz from MIMIC-CXR 2.1.0 (reader axis, optional)
MIMIC_TARGET = "Pleural Effusion"   # finding the image view (V2) is trained on
MIMIC_N_CASES = 4000       # studies sampled for the image-based EM (D2c)
RUN_MIMIC_ARCH = True      # D2b: run the Section II architecture on MIMIC pairs (about 1 h on CPU)
VINDR_DIR = None           # physionet.org/content/vindr-cxr/1.0.0  (annotations/*.csv, train/*.dicom), or the authors' Kaggle
                           # competition copy (train.csv, train/*.dicom; join the competition and check its rules allow
                           # publication use): https://www.kaggle.com/competitions/vinbigdata-chest-xray-abnormalities-detection
VINDR_WITH_IMAGES = True   # D3d needs the DICOMs and pydicom
REFLACX_DIR = None         # physionet.org/content/reflacx-xray-localization/1.0.0  (unzipped main_data.zip)
CHEXPERT_DIR = None        # Stanford AIMI CheXpert-v1.0 or -v1.0-small (train.csv, valid.csv, images). A Kaggle re-upload by
                           # an individual (https://www.kaggle.com/datasets/ashery/chexpert) is fine for testing the code; for
                           # results in the paper, download from Stanford AIMI under its research use agreement.
CHEXPERT_WITH_IMAGES = True
CORBENCHX_PATH = None      # CorBenchX release (file or folder of CSV/JSON/JSONL); ReXErr-v1 also works
# Part E (open or easier-access alternatives, no PhysioNet credentialing). Leave None to skip.
REXGRADIENT_DIR = None         # huggingface.co/datasets/rajpurkarlab/ReXGradient-160K (licence click-through)
REXGRADIENT_IMAGES_DIR = None  # the extracted deid_png folder (cat deid_png.part* > deid_png.tar; tar -xf deid_png.tar)
REX_LABELER = "auto"           # "auto" (CheXbert if installed, else rules), "chexbert" or "lexical"
REX_MAX_LABEL = 20000          # studies labelled (whole patients); CheXbert is slow on CPU
RUN_ALT_ARCH = True            # E1b: the Section II architecture on ReXGradient pairs (about 1 h on CPU)
# E2 reuses NIH_LABELS_DIR, and E3 reuses IU_DATA_DIR (or the ReXGradient reports from E1).

import os
os.environ["OMP_NUM_THREADS"] = "1"      # set before numpy/torch import: deterministic and ~10x faster
os.environ["MKL_NUM_THREADS"] = "1"
RES = os.path.join(OUT_DIR, "results"); FIG = os.path.join(OUT_DIR, "figs"); HERE = OUT_DIR
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)

if QUICK:
    SEEDS = dict(A=range(2), B=range(100, 101), C=range(200, 201), D=range(300, 301),
                 E=range(400, 401), F=range(2), G=range(2))
    M4_SEEDS, M4_RESTARTS, REVIEW_SEEDS = range(2), 8, range(2)
else:   # exactly the protocol reported in the paper (145 runs)
    SEEDS = dict(A=range(10), B=range(100, 105), C=range(200, 205), D=range(300, 305),
                 E=range(400, 405), F=range(10), G=range(10))
    M4_SEEDS, M4_RESTARTS, REVIEW_SEEDS = range(10), 30, range(10)
print("mode:", "QUICK" if QUICK else "FULL", "| outputs in", os.path.abspath(OUT_DIR))

# ---- running on Kaggle: fill any path left as None from the datasets attached with "Add Input" ----
# Only datasets that have a legitimate Kaggle copy are looked for. NIH_LABELS_DIR (Google request form) and
# REXGRADIENT_DIR (Hugging Face licence) are not on Kaggle: upload them as a PRIVATE Kaggle dataset and they are
# found too. Never attach Kaggle copies of MIMIC-CXR or other PhysioNet data: that breaks the data use agreement.
KAGGLE_INPUT = "/kaggle/input"
KAGGLE_SIGNATURES = {     # setting -> (file or folder that identifies the dataset; the path set is the folder holding it,
    "IU_DATA_DIR": "indiana_reports.csv",                      #  except deid_png, which is the image folder itself)
    "NIH_IMAGES_DIR": "Data_Entry_2017.csv",
    "NIH_LABELS_DIR": "individual_readers.csv",
    "VINDR_DIR": "sample_submission.csv",                      # VinBigData competition (train.csv, train/, test/)
    "CHEXPERT_DIR": "CheXpert-v1.0-small",
    "REXGRADIENT_DIR": "interstitial_pattern_bbox.json",
    "REXGRADIENT_IMAGES_DIR": "deid_png",
}

def _kaggle_scan(base, max_depth=5):
    # map each signature name to where it was found, without walking into large image folders
    found = {}
    names = set(KAGGLE_SIGNATURES.values())
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        for n in list(dirs) + files:
            if n in names and n not in found:
                found[n] = (root, os.path.join(root, n))
        if depth >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [d for d in dirs if not d.lower().startswith(("images_", "train", "test", "patient", "p1", "files"))
                       or d in names]
    return found

if os.path.isdir(KAGGLE_INPUT):
    _found = _kaggle_scan(KAGGLE_INPUT)
    for _var, _sig in KAGGLE_SIGNATURES.items():
        if globals().get(_var) is None and _sig in _found:
            _root, _path = _found[_sig]
            globals()[_var] = _path if _sig == "deid_png" else _root
            print(f"Kaggle input found: {_var} = {globals()[_var]}")
    if not _found:
        print("Running on Kaggle, but no known dataset is attached (Add Input); Parts C-E will skip.")

In [ ]:
import json, sys, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, balanced_accuracy_score)
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from IPython.display import Image, display
torch.set_num_threads(1)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

def show(*names):
    for n in names:
        p = os.path.join(FIG, n)
        if os.path.exists(p):
            display(Image(filename=p, width=640))

In [ ]:
os.makedirs("realdata/results", exist_ok=True)
if os.path.abspath("realdata") not in sys.path:
    sys.path.insert(0, os.path.abspath("realdata"))
RD_OUT = os.path.abspath("realdata/results")
D_STATUS = {}

def run_d(step, name, path, fn):
    # run one Part D step; record run / skipped / failed without stopping the notebook
    if not RUN_PART_D or not path:
        D_STATUS[step] = (name, "skipped (path not set)")
        print(f"{step} {name}: skipped. Set its path in the configuration cell.")
        return None
    if not os.path.exists(path):
        D_STATUS[step] = (name, f"skipped (not found: {path})")
        print(f"{step} {name}: path not found: {path}")
        return None
    try:
        out = fn()
        D_STATUS[step] = (name, "run")
        return out
    except SystemExit as e:          # the runners stop with a readable message on a schema problem
        D_STATUS[step] = (name, f"failed: {e}")
        print(f"{step} {name} stopped: {e}")
    except Exception as e:
        D_STATUS[step] = (name, f"failed: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
    return None

def show_md(fn):
    p = os.path.join(RD_OUT, fn)
    if os.path.exists(p):
        from IPython.display import Markdown
        display(Markdown(open(p).read()))

In [ ]:
%%writefile realdata/rd_common.py
"""
Shared helpers for the Part D real-dataset runners (Table VII of the manuscript).

Every runner follows the same rules:
  * it reads only files you have downloaded yourself under the dataset's own licence
    (PhysioNet credentialed access for MIMIC-CXR, RadGraph, VinDr-CXR, REFLACX and
    CorBenchX; Stanford AIMI registration for CheXpert);
  * it finds files by their official names anywhere under the directory you give it,
    prints the columns it found before using them, and stops with a clear message if
    a required file or column is missing;
  * it never treats a labeler output as ground truth for the cause of a disagreement:
    no dataset here labels that cause, so every check is a face-validity or
    assumption check, stated as such in the output.
"""
import gzip
import json
import os
import re
import sys

import numpy as np
import pandas as pd

# ------------------------------------------------------------------ files
def find_file(root, *names):
    """First file under root whose basename matches one of names (case-insensitive;
    a name ending in .csv also matches .csv.gz)."""
    want = set()
    for n in names:
        want.add(n.lower())
        if n.lower().endswith(".csv"):
            want.add(n.lower() + ".gz")
    for dp, _, fns in os.walk(root):
        for fn in sorted(fns):
            if fn.lower() in want:
                return os.path.join(dp, fn)
    return None


def find_files(root, pattern):
    rx = re.compile(pattern, re.I)
    out = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if rx.search(fn):
                out.append(os.path.join(dp, fn))
    return sorted(out)


def read_table(path, **kw):
    df = pd.read_csv(path, **kw)
    print(f"  {os.path.basename(path)}: {len(df)} rows; columns = {list(df.columns)}")
    return df


def require(path, what):
    if path is None:
        sys.exit(f"{what} not found. Check the directory you set in the configuration cell.")
    return path


def col(df, *cands, required=True):
    """Column of df matching one of cands (case-insensitive, ignoring _ and spaces)."""
    norm = lambda s: re.sub(r"[\s_]+", "", str(s).lower())
    m = {norm(c): c for c in df.columns}
    for c in cands:
        if norm(c) in m:
            return m[norm(c)]
    if required:
        sys.exit(f"none of the columns {cands} found; columns are {list(df.columns)}")
    return None


def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("wrote", path)


def save_md(lines, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print("wrote", path)


def md_table(df, floatfmt="{:.3f}"):
    cols = list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for r in df.to_dict("records"):
        cells = [floatfmt.format(v) if isinstance(v, (float, np.floating)) and np.isfinite(v)
                 else ("--" if isinstance(v, (float, np.floating)) else str(v)) for v in r.values()]
        out.append("| " + " | ".join(cells) + " |")
    return out


# ------------------------------------------------------------------ report text
SECTION_RX = re.compile(r"^\s*(FINDINGS|IMPRESSION|CONCLUSION)\s*:", re.I | re.M)


def report_sections(text):
    """Findings + impression of a MIMIC-style free-text report (falls back to the whole text)."""
    text = "" if text is None else str(text)
    parts = {}
    hits = list(re.finditer(r"^\s*([A-Z][A-Z /&]+):", text, re.M))
    for i, h in enumerate(hits):
        name = h.group(1).strip().upper()
        end = hits[i + 1].start() if i + 1 < len(hits) else len(text)
        parts[name] = text[h.end():end]
    body = " ".join(parts.get(k, "") for k in ("FINDINGS", "IMPRESSION", "CONCLUSION")).strip()
    if not body:
        body = text
    body = re.sub(r"_{2,}", " ", body)           # MIMIC de-identification placeholders
    return re.sub(r"\s+", " ", body).strip()


# Hedge cues for real reports (the synthetic harness uses its own template cues).
HEDGE_CUES = [
    r"\bpossibl[ey]\b", r"\bprobabl[ey]\b", r"\blikely\b", r"\bmay\b", r"\bmight\b", r"\bcould\b",
    r"\bcannot (?:be )?(?:excluded|exclude|ruled out|rule out)\b", r"\bnot (?:be )?excluded\b",
    r"\bsuggest(?:s|ive|ing)?\b", r"\bsuspicious\b", r"\bquestionabl[ey]\b", r"\bequivocal\b",
    r"\bindeterminate\b", r"\bconcerning for\b", r"\bversus\b", r"\bvs\.?\b", r"\bpresumabl[ey]\b",
    r"\bdifficult to (?:exclude|assess)\b", r"\bpotential(?:ly)?\b", r"\bperhaps\b",
]
HEDGE_RX = re.compile("|".join(HEDGE_CUES), re.I)
NEG_RX = re.compile(r"\b(no|without|negative for|free of|absence of|resolved|not seen|no evidence of)\b", re.I)

FINDING_TERMS = {
    "Atelectasis": ["atelecta"],
    "Cardiomegaly": ["cardiomegaly", "enlarged cardiac silhouette", "heart size is enlarged",
                     "enlarged heart", "cardiac enlargement"],
    "Consolidation": ["consolidat"],
    "Edema": ["edema", "vascular congestion"],
    "Pleural Effusion": ["effusion"],
    "Pneumothorax": ["pneumothora"],
    "Pneumonia": ["pneumonia"],
    "Lung Opacity": ["opacit", "airspace disease", "infiltrat"],
    "Lung Lesion": ["nodul", "mass", "lesion"],
    "Fracture": ["fracture"],
}
ANY_FINDING = sorted({t for v in FINDING_TERMS.values() for t in v})


def sentences(text):
    return [s for s in re.split(r"(?<=[.;])\s+|\n", str(text)) if s.strip()]


def lexical_v1(text, terms=None):
    """(hedge, mention) for one report, with the paper's View 1 definitions on real text:
    hedge   = 1 if a sentence that names a finding also carries a hedge cue;
    mention = 1 if a sentence names a finding without a negation cue (asserted present)."""
    terms = terms or ANY_FINDING
    hedge = mention = 0
    for s in sentences(text):
        low = s.lower()
        if not any(t in low for t in terms):
            continue
        if HEDGE_RX.search(low):
            hedge = 1
        elif not NEG_RX.search(low):
            mention = 1
    return float(hedge), float(mention)


# ------------------------------------------------------------------ images
def laplacian_var(img):
    lap = np.abs(np.gradient(np.gradient(img, axis=0), axis=0)
                 + np.gradient(np.gradient(img, axis=1), axis=1))
    return float(lap.var())


def load_gray(path, size=32):
    """Grayscale image in [0,1], resized to size x size. JPG/PNG via PIL, DICOM via pydicom."""
    from PIL import Image
    if path.lower().endswith((".dcm", ".dicom")):
        try:
            import pydicom
        except ImportError:
            sys.exit("pydicom is needed for DICOM images: pip install pydicom")
        ds = pydicom.dcmread(path)
        a = ds.pixel_array.astype(np.float32)
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            a = a.max() - a
        a = (a - a.min()) / (a.max() - a.min() + 1e-8)
        im = Image.fromarray((a * 255).astype(np.uint8))
    else:
        im = Image.open(path).convert("L")
    im = im.resize((size, size), Image.BILINEAR)
    return np.asarray(im, dtype=np.float32) / 255.0


def index_images(root, exts=(".jpg", ".jpeg", ".png", ".dcm", ".dicom")):
    idx = {}
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(exts):
                idx.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))
    return idx


# ------------------------------------------------------------------ statistics
def auc(y, s):
    from sklearn.metrics import roc_auc_score
    y = np.asarray(y); s = np.asarray(s, dtype=float)
    ok = np.isfinite(s)
    if ok.sum() < 4 or len(np.unique(y[ok])) < 2:
        return float("nan")
    return float(roc_auc_score(y[ok], s[ok]))


def cohen_kappa(a, b):
    from sklearn.metrics import cohen_kappa_score
    a = np.asarray(a); b = np.asarray(b)
    if len(np.unique(np.concatenate([a, b]))) < 2:
        return float("nan")
    return float(cohen_kappa_score(a, b))


def fleiss_kappa(counts):
    """counts: (n_items, n_categories) array of rater counts per item (same raters per item)."""
    counts = np.asarray(counts, dtype=float)
    n = counts.sum(1)
    counts, n = counts[n >= 2], n[n >= 2]
    if len(counts) == 0:
        return float("nan")
    p_j = counts.sum(0) / counts.sum()
    P_i = ((counts ** 2).sum(1) - n) / (n * (n - 1))
    Pbar, Pe = P_i.mean(), (p_j ** 2).sum()
    return float((Pbar - Pe) / (1 - Pe)) if Pe < 1 else float("nan")


def logit_or(y, X, names):
    """Odds ratios (95% Wald CI) from an unpenalized logistic regression; statsmodels if present."""
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float)
    try:
        import statsmodels.api as sm
        m = sm.Logit(y, sm.add_constant(X, has_constant="add")).fit(disp=0)
        b, se, p = m.params[1:], m.bse[1:], m.pvalues[1:]
    except Exception:
        from sklearn.linear_model import LogisticRegression
        m = LogisticRegression(C=1e6, max_iter=2000).fit(X, y)
        b = m.coef_[0]
        pr = m.predict_proba(X)[:, 1]
        Xd = np.hstack([np.ones((len(X), 1)), X])
        cov = np.linalg.pinv(Xd.T @ (Xd * (pr * (1 - pr))[:, None]))
        se = np.sqrt(np.diag(cov))[1:]
        from scipy.stats import norm
        p = 2 * norm.sf(np.abs(b / se))
    return pd.DataFrame({"term": names, "odds_ratio": np.exp(b), "ci_low": np.exp(b - 1.96 * se),
                         "ci_high": np.exp(b + 1.96 * se), "p": p})


def dawid_skene(L, n_iter=50, n_classes=2, smooth=1.0):
    """Dawid-Skene EM. L: (n_items, n_readers) int labels in {0..C-1}, -1 = not read.
    Returns posterior (n_items, C), class prior, confusion (n_readers, C, C)."""
    L = np.asarray(L)
    n, R = L.shape
    C = n_classes
    T = np.zeros((n, C))                      # init: majority vote
    for c in range(C):
        T[:, c] = (L == c).sum(1)
    T = (T + 1e-3) / (T + 1e-3).sum(1, keepdims=True)
    for _ in range(n_iter):
        prior = T.mean(0)
        conf = np.full((R, C, C), smooth)
        for r in range(R):
            m = L[:, r] >= 0
            for l in range(C):
                conf[r, :, l] += T[m][L[m, r] == l].sum(0)
        conf /= conf.sum(2, keepdims=True)
        logT = np.tile(np.log(prior + 1e-12), (n, 1))
        for r in range(R):
            m = L[:, r] >= 0
            logT[m] += np.log(conf[r][:, L[m, r]].T + 1e-12)
        logT -= logT.max(1, keepdims=True)
        T = np.exp(logT); T /= T.sum(1, keepdims=True)
    return T, prior, conf


# ------------------------------------------------------------------ CheXpert-style labels from text
# Used when the CheXbert model is not available.
CHEXPERT14 = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
              "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
              "Pleural Other", "Fracture", "Support Devices", "No Finding"]
TERMS_EN = dict(FINDING_TERMS, **{
    "Enlarged Cardiomediastinum": ["mediastinal widening", "widened mediastinum", "enlarged cardiomediastinal",
                                   "mediastinum is widened", "cardiomediastinal silhouette is enlarged"],
    "Pleural Other": ["pleural thickening", "pleural plaque", "fibrothorax"],
    "Support Devices": ["tube", "catheter", "line", "pacemaker", "wire", "stent", "device", "port"],
})

def finding_status(text, terms):
    """CheXpert-style status of one finding in a report: 1 present, -1 uncertain, 0 negated, nan unmentioned.
    Aggregation as in the CheXpert labeler: present beats uncertain beats negated."""
    st = set()
    for s in sentences(text):
        low = s.lower()
        if not any(t in low for t in terms):
            continue
        if HEDGE_RX.search(low):
            st.add(-1)
        elif NEG_RX.search(low):
            st.add(0)
        else:
            st.add(1)
    if 1 in st:
        return 1.0
    if -1 in st:
        return -1.0
    if 0 in st:
        return 0.0
    return float("nan")


def lexical_labels(text):
    out = {f: finding_status(text, TERMS_EN[f]) for f in CHEXPERT14 if f in TERMS_EN}
    pos = [v for k, v in out.items() if k != "Support Devices" and v == 1]
    out["No Finding"] = float("nan") if pos else 1.0
    return out


def radgraph_status(label):
    """Map RadGraph (OBS-DP / OBS-U / OBS-DA) and RadGraph-XL ('Observation::definitely present', ...)
    labels to one of OBS-DP, OBS-U, OBS-DA, or None for anatomy."""
    l = str(label)
    if l.startswith("OBS-"):
        return l[:6] if l[:6] in ("OBS-DP", "OBS-DA") else ("OBS-U" if l.startswith("OBS-U") else None)
    if l.lower().startswith("observation"):
        if "uncertain" in l:
            return "OBS-U"
        if "absent" in l:
            return "OBS-DA"
        if "present" in l:
            return "OBS-DP"
    return None

In [ ]:
%%writefile realdata/corbenchx_run.py
"""
D6 - CorBenchX (Zou et al., 2025, arXiv:2505.12057): 26,326 MIMIC-CXR reports with
LLM-injected errors of five types (omission, insertion, spelling error, side confusion,
other), each paired with the clean report. The authors state the data were submitted to
PhysioNet; access follows MIMIC-CXR credentialing. ReXErr (PhysioNet "rexerr-v1") has the
same clean/erroneous-pair structure and can be run with the same code via --field-* options.

Role in Table VII: synthetic-error baseline comparison for triage. The clean report is used as
the second read r' of the same study, so each pair exercises the reader-reader axis with a
known injected error:

  (a) Which injected errors can the paper's text view see at all? Per error type, the share of
      pairs whose View 1 indicators (hedge, mention) differ between the clean and the
      erroneous report, and the direction of the change.
  (b) Cause assigned by the paper's rule over (r, r') vs the injected type mapped onto the
      taxonomy: omission -> omission; spelling / other -> style (wording only); insertion and
      side confusion -> outside the taxonomy (a confident factual conflict, Table I last row).
      Rule: r hedges where r' does not -> hedge; r' mentions a finding that r does not ->
      omission; r mentions a finding r' does not -> conflict; text differs otherwise -> style.
  (c) Triage (Algorithm 2) on the assigned cause: share of each injected type sent to review.
      Errors outside the taxonomy that the 4-class system must place on hedge or style are
      auto-corrected without review: this measures that failure mode, stated in Section II-E.

FILES: one CSV / JSON / JSONL file (or a directory of them) with one row per erroneous report.
Columns are discovered (printed) and can be set explicitly:
  --field-clean  (default: first column whose name contains original/clean/gt/reference)
  --field-error  (default: first column whose name contains error/corrupt/noisy + report/text)
  --field-type   (default: first column whose name contains type; several types may be
                  separated by ; , or | or given as a list)

Also runs on the open pairs written by inject_errors.py (E3), which add a "hedge" type.

USAGE  python3 corbenchx_run.py --corbenchx-path <file or dir> [--prefix corbenchx]
"""
import argparse
import json
import os
import re
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C

TYPE_MAP = {"omission": "omission", "insertion": "conflict", "side confusion": "conflict",
            "side": "conflict", "laterality": "conflict", "spelling error": "style", "spelling": "style",
            "other": "style", "hedge": "hedge"}
CAUSES = ["faithful", "omission", "hedge", "style", "conflict"]


def _read_any(p):
    if p.endswith(".jsonl"):
        return pd.read_json(p, lines=True)
    if p.endswith(".json"):
        d = json.load(open(p))
        if isinstance(d, dict):
            d = list(d.values()) if all(isinstance(v, dict) for v in d.values()) else d.get("data", d)
        return pd.json_normalize(d)
    return pd.read_csv(p)


def load(path, fc, fe, ft):
    files = [path] if os.path.isfile(path) else C.find_files(path, r"\.(csv|json|jsonl)$")
    if not files:
        sys.exit("no CSV/JSON/JSONL found at " + path)
    d = pd.concat([_read_any(p) for p in files], ignore_index=True)
    print(f"CorBenchX: {len(d)} rows from {len(files)} file(s); columns = {list(d.columns)}")
    low = {c: c.lower() for c in d.columns}
    pick = lambda pats, excl=(): next((c for c, l in low.items() if any(re.search(p, l) for p in pats)
                                        and not any(re.search(x, l) for x in excl)), None)
    fc = fc or pick([r"original", r"clean", r"\bgt\b", r"ground", r"reference"], excl=[r"span", r"type"])
    fe = fe or pick([r"(error|corrupt|noisy|erroneous).*(report|text)", r"(report|text).*(error|corrupt|noisy)"],
                    excl=[r"span", r"type", r"desc"])
    ft = ft or pick([r"type"])
    print(f"  clean report: {fc}; erroneous report: {fe}; error type: {ft}")
    if not (fc and fe and ft):
        sys.exit("set --field-clean / --field-error / --field-type to the column names printed above")
    d = d.rename(columns={fc: "clean", fe: "error", ft: "etype"})
    d["etype"] = d.etype.map(lambda v: [str(x).strip().lower() for x in (v if isinstance(v, list)
                                         else re.split(r"[;,|]", str(v))) if str(x).strip()])
    return d


def rule(clean, err):
    hc, mc = C.lexical_v1(clean); he, me = C.lexical_v1(err)
    if he and not hc:
        return "hedge"          # checked first: hedging an asserted finding also removes the assertion
    if mc and not me:
        return "omission"
    if me and not mc:
        return "conflict"
    if clean.strip() != err.strip():
        return "style"          # wording, spelling or punctuation differs; View 1 unchanged
    return "faithful"


def algorithm2(cause):
    return "review" if cause == "omission" else "auto"          # conf = 1 for a deterministic rule


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--corbenchx-path", required=True)
    ap.add_argument("--field-clean"); ap.add_argument("--field-error"); ap.add_argument("--field-type")
    ap.add_argument("--out", default="results")
    ap.add_argument("--prefix", default="corbenchx")
    ap.add_argument("--title", default="D6 CorBenchX")
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    d = load(a.corbenchx_path, a.field_clean, a.field_error, a.field_type)
    v = np.array([C.lexical_v1(str(c)) + C.lexical_v1(str(e)) for c, e in zip(d.clean, d.error)])
    d["d_hedge"] = v[:, 2] - v[:, 0]; d["d_mention"] = v[:, 3] - v[:, 1]
    d["cause"] = [rule(str(c), str(e)) for c, e in zip(d.clean, d.error)]
    d["route"] = d.cause.map(algorithm2)
    single = d[d.etype.map(len) == 1].copy()
    single["etype1"] = single.etype.map(lambda x: x[0])
    single["mapped"] = single.etype1.map(lambda t: TYPE_MAP.get(t, "style"))
    vis = single.groupby("etype1").agg(n=("cause", "size"),
                                       v1_changed=("d_mention", lambda s: float(((s != 0) | (single.loc[s.index, "d_hedge"] != 0)).mean())),
                                       mention_lost=("d_mention", lambda s: float((s < 0).mean())),
                                       mention_gained=("d_mention", lambda s: float((s > 0).mean())),
                                       hedge_gained=("d_hedge", lambda s: float((s > 0).mean())),
                                       to_review=("route", lambda s: float((s == "review").mean()))).reset_index()
    conf = pd.crosstab(single.mapped, pd.Categorical(single.cause, CAUSES), dropna=False)
    recall = {m: float(conf.loc[m, m] / conf.loc[m].sum()) for m in conf.index if m in conf.columns and conf.loc[m].sum()}
    res = dict(n=len(d), n_single_error=len(single), recall_by_mapped_cause=recall,
               conflict_auto_corrected=float((single[single.mapped == "conflict"].route == "auto").mean())
               if (single.mapped == "conflict").any() else np.nan)
    vis.to_csv(os.path.join(a.out, f"{a.prefix}_visibility.csv"), index=False)
    conf.to_csv(os.path.join(a.out, f"{a.prefix}_confusion.csv"))
    C.save_json(res, os.path.join(a.out, f"{a.prefix}_results.json"))
    L = [f"# {a.title} ({len(d)} erroneous reports, {len(single)} with a single injected error)", "",
         "**(a, c) Visibility of each injected error to View 1, and Algorithm 2 routing**", ""]
    L += C.md_table(vis)
    cf = conf.copy(); cf.insert(0, "injected (mapped)", cf.index)
    L += ["", "**(b) Cause from the rule over (r, r') vs injected type mapped to the taxonomy**", ""]
    L += C.md_table(cf.reset_index(drop=True))
    L += ["", "* recall per mapped cause: " + ", ".join(f"{k} {v:.3f}" for k, v in recall.items()),
          f"* injected conflicts (insertion, side confusion) auto-corrected without review: {res['conflict_auto_corrected']:.3f}"]
    C.save_md(L, os.path.join(a.out, f"{a.prefix}_summary.md"))
    return res


if __name__ == "__main__":
    main()

In [ ]:
%%writefile realdata/mimic_run.py
"""
D2 - MIMIC-CXR (Johnson et al., 2019) with MIMIC-CXR-JPG; PhysioNet credentialed access.

Role in Table VII: primary source for the image-report and current-prior axes. This runner
executes the real-data protocol of Section III-C (formerly "implemented, not executed"):

  (a) Concomitant link, labels only. Does the CheXpert labeler's "uncertain" (-1) rate depend
      on acquisition (AP vs PA, and image sharpness when images are given)? The model
      assumes the class prior depends on quality (Eq. 17); this checks that assumption on
      real reports. Logistic regression per finding and pooled with finding fixed effects.
  (b) Current-prior axis, labels only. Consecutive studies of the same patient (ordered by
      StudyDate/StudyTime): how often does a finding flip between present and absent, and
      how often is an uncertain label at t resolved to a definite one at t+1? A hedge that is
      later resolved is consistent with genuine image ambiguity at time t.
  (c) The latent-class EM of Part A on real views (needs images; optional RadGraph and
      provider list), with label-free naming. No cause labels exist, so only face-validity
      checks are reported: does the component named "hedge" concentrate on studies the
      CheXpert labeler marked uncertain for the target finding; is the class mix stable
      across quality terciles; how overconfident are the posteriors.
  (d) Writes pairs.csv (filename, patientid, txt, y_target) so the architecture runner of
      Part C1 can be run on MIMIC with --mode pairs.

FILES (MIMIC-CXR-JPG 2.0.0/2.1.0 and MIMIC-CXR 2.0.0/2.1.0, any layout under the dirs given)
  mimic-cxr-2.0.0-metadata.csv(.gz)  dicom_id, subject_id, study_id, ViewPosition, StudyDate, StudyTime
  mimic-cxr-2.0.0-chexpert.csv(.gz)  subject_id, study_id, 14 findings in {1, 0, -1, blank}
  files/pXX/pXXXXXXXX/sXXXXXXXX/<dicom_id>.jpg    (images, optional: needed for (c))
  files/pXX/pXXXXXXXX/sXXXXXXXX.txt               (reports from mimic-cxr-reports.zip)
  cxr-provider-list.csv(.gz)  (MIMIC-CXR 2.1.0, optional: reader axis View 3)
  radgraph_v1_mimic.csv       (written by D1 from MIMIC-CXR_graphs.json, optional)

USAGE
  python3 mimic_run.py --jpg-dir <mimic-cxr-jpg> --reports-dir <mimic-cxr> [--provider-csv f]
                       [--radgraph-features f] [--target "Pleural Effusion"] [--n-cases 4000]
"""
import argparse
import os
import re
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C

FINDINGS = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
            "Fracture", "Lung Lesion", "Lung Opacity", "Pleural Effusion", "Pleural Other",
            "Pneumonia", "Pneumothorax", "Support Devices"]
CLASS_NAMES = ["faithful", "omission", "hedge", "style"]


# ------------------------------------------------------------------ loading
def load_tables(jpg_dir):
    print("MIMIC-CXR-JPG tables:")
    meta = C.read_table(C.require(C.find_file(jpg_dir, "mimic-cxr-2.0.0-metadata.csv"), "metadata csv"))
    chx = C.read_table(C.require(C.find_file(jpg_dir, "mimic-cxr-2.0.0-chexpert.csv"), "chexpert csv"))
    for c in ("dicom_id", "subject_id", "study_id", "ViewPosition"):
        C.col(meta, c)
    fnd = [f for f in FINDINGS if f in chx.columns]
    print(f"  findings present: {fnd}")
    front = meta[meta.ViewPosition.isin(["PA", "AP"])].copy()
    if "StudyDate" in front.columns:
        t = front.get("StudyTime", pd.Series(0, index=front.index)).fillna(0).astype(float)
        front["order"] = front.StudyDate.astype(float) * 1e6 + t
    else:
        front["order"] = front.study_id
    front = front.sort_values(["study_id", "ViewPosition"], ascending=[True, False])  # PA before AP
    st = front.drop_duplicates("study_id")[["subject_id", "study_id", "dicom_id", "ViewPosition", "order"]]
    st = st.merge(chx[["study_id"] + fnd], on="study_id", how="inner")
    print(f"  frontal studies with CheXpert labels: {len(st)} ({st.subject_id.nunique()} patients)")
    return st, fnd


def attach_reports(st, reports_dir):
    idx = {}
    for p in C.find_files(reports_dir, r"^s\d+\.txt$"):
        idx[int(os.path.basename(p)[1:-4])] = p
    print(f"  report files found: {len(idx)}")
    st = st[st.study_id.isin(idx)].copy()
    st["txt"] = [C.report_sections(open(idx[s], errors="ignore").read()) for s in st.study_id]
    return st[st.txt.str.len() > 0]


def attach_provider(st, path):
    if not path or not os.path.exists(path):
        print("  provider list not given: View 3 (reader) will be omitted")
        st["reader_id"] = np.nan
        return st
    p = C.read_table(path)
    sid = C.col(p, "study_id")
    pc = next((c for c in p.columns if re.search(r"provider|attending|author|reader", c, re.I)), None)
    if pc is None:
        print("  no provider column found: View 3 omitted"); st["reader_id"] = np.nan; return st
    print(f"  reader column: {pc}")
    return st.merge(p[[sid, pc]].rename(columns={sid: "study_id", pc: "reader_id"}).drop_duplicates("study_id"),
                    on="study_id", how="left")


def attach_images(st, jpg_dir, size=32):
    idx = C.index_images(jpg_dir, exts=(".jpg", ".jpeg", ".png"))
    keep = st.dicom_id.isin(idx)
    print(f"  images found for {keep.sum()} / {len(st)} sampled studies")
    st = st[keep].copy()
    st["img"] = [C.load_gray(idx[d], size) for d in st.dicom_id]
    st["q_meas"] = st.img.map(C.laplacian_var)
    st["img_path"] = [idx[d] for d in st.dicom_id]
    return st


# ------------------------------------------------------------------ (a) concomitant link
def concomitant(st, fnd, quality=None):
    rows, pooled = [], []
    for f in fnd:
        if f == "Support Devices":
            continue
        m = st[f].notna()                       # finding mentioned at all
        y = (st.loc[m, f] == -1).astype(int).values
        if m.sum() < 30 or y.sum() < 5 or y.sum() == len(y):
            continue
        X = [(st.loc[m, "ViewPosition"] == "AP").astype(float).values]
        names = ["AP_vs_PA"]
        if quality is not None:
            q = quality[m.values]
            X.append((q - q.mean()) / (q.std() + 1e-8)); names.append("sharpness_z")
        o = C.logit_or(y, np.column_stack(X), names)
        o.insert(0, "finding", f); o.insert(1, "n_mentioned", int(m.sum())); o.insert(2, "uncertain_rate", y.mean())
        rows.append(o)
        pooled.append(pd.DataFrame({"y": y, "ap": X[0], "finding": f}))
    per = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    pool = None
    if pooled:
        P = pd.concat(pooled, ignore_index=True)
        D = pd.get_dummies(P.finding, drop_first=True).astype(float)
        o = C.logit_or(P.y.values, np.column_stack([P.ap.values, D.values]), ["AP_vs_PA"] + list(D.columns))
        pool = o.iloc[0].to_dict()
        pool["n"] = len(P)
        pool["rate_AP"] = P.y[P.ap == 1].mean(); pool["rate_PA"] = P.y[P.ap == 0].mean()
    return per, pool


# ------------------------------------------------------------------ (b) current-prior axis
def longitudinal(st, fnd):
    s = st.sort_values(["subject_id", "order"])
    rows = []
    for f in fnd:
        a = s[f].values; sub = s.subject_id.values
        same = sub[1:] == sub[:-1]
        prev, cur = a[:-1][same], a[1:][same]
        both_def = np.isin(prev, [0, 1]) & np.isin(cur, [0, 1])
        unc = prev == -1
        rows.append(dict(finding=f, n_pairs=int(same.sum()),
                         flip_rate_definite=float((prev[both_def] != cur[both_def]).mean()) if both_def.any() else np.nan,
                         n_uncertain_at_t=int(unc.sum()),
                         resolved_present=float((cur[unc] == 1).mean()) if unc.any() else np.nan,
                         resolved_absent_or_unmentioned=float(((cur[unc] == 0) | pd.isna(cur[unc])).mean()) if unc.any() else np.nan,
                         still_uncertain=float((cur[unc] == -1).mean()) if unc.any() else np.nan))
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ (c) EM on real views
def em_views(st, target, seed, rg_path, cnn_epochs, harness):
    df = st.copy().reset_index(drop=True)
    if rg_path and os.path.exists(rg_path):
        rg = pd.read_csv(rg_path, dtype={"study_id": str})
        df = df.assign(study_id=df.study_id.astype(str)).merge(rg, on="study_id", how="left")
        ok = df.rg_hedge.notna()
        print(f"  View 1 from RadGraph inference for {ok.sum()} / {len(df)} studies (lexical for the rest)")
    else:
        ok = pd.Series(False, index=df.index)
        print("  View 1: lexical proxy (no RadGraph features given)")
    terms = C.TERMS_EN.get(target)
    lex = np.array([C.lexical_v1(t, terms) for t in df.txt])
    df["view1_hedge"] = np.where(ok, df.get("rg_hedge", 0), lex[:, 0])
    df["view1_mention"] = np.where(ok, df.get("rg_mention", 0), lex[:, 1])
    feat = ["view1_hedge", "view1_mention"]
    # View 2: cross-fitted CNN for the target finding, residualised on quality
    df["true_finding"] = (df[target] == 1).astype(np.float32)   # labeler label; used only to train V2
    v2, _ = harness.build_view2_crossfit(df, seed, epochs=cnn_epochs)
    from sklearn.linear_model import LinearRegression
    df["view2_raw"] = v2
    df["view2_resid"] = v2 - LinearRegression().fit(df[["q_meas"]], v2).predict(df[["q_meas"]])
    feat.append("view2_resid")
    # View 3: provider leave-one-out rates of observed hedge and non-mention
    if df.reader_id.notna().sum() > 0:
        cnt = df.groupby("reader_id").reader_id.transform("count")
        valid = df.reader_id.notna() & (cnt >= harness.MIN_CASES_PER_READER)
        for src, out in (("view1_hedge", "view3_hedge_obs"), ("view1_mention", "view3_nomention_obs")):
            flag = df[src] if src == "view1_hedge" else 1 - df[src]
            s_ = flag.groupby(df.reader_id).transform("sum"); c_ = flag.groupby(df.reader_id).transform("count")
            df[out] = (s_ - flag) / (c_ - 1).clip(lower=1)
            df.loc[~valid, out] = df.loc[valid, out].mean() if valid.any() else flag.mean()
        feat += ["view3_hedge_obs", "view3_nomention_obs"]
        print(f"  View 3: {valid.sum()} studies with a provider having >= {harness.MIN_CASES_PER_READER} studies")
    return df, feat


def em_face_validity(df, feat, target, seed, harness, n_restarts):
    pred_raw, resp, ll = harness.fit_em(df, feat, seed, K=4, n_restarts=n_restarts)
    mapping = harness.anchor_mapping(resp, df.view1_hedge.values, df.view1_mention.values)
    inv = {v: k for k, v in mapping.items()}
    named = np.array([inv[k] for k in pred_raw])
    post = resp[:, [mapping[c] for c in range(4)]]          # columns in CLASS_NAMES order
    unc = (df[target] == -1).values
    q3 = pd.qcut(df.q_meas.rank(method="first"), 3, labels=["low", "mid", "high"])
    mix = pd.crosstab(q3, pd.Categorical([CLASS_NAMES[k] for k in named], CLASS_NAMES), normalize="index")
    res = dict(
        n=len(df), features=feat, log_lik=float(ll),
        class_share={CLASS_NAMES[k]: float((named == k).mean()) for k in range(4)},
        labeler_uncertain_rate=float(unc.mean()),
        p_uncertain_given_named_hedge=float(unc[named == 2].mean()) if (named == 2).any() else np.nan,
        p_uncertain_given_other=float(unc[named != 2].mean()) if (named != 2).any() else np.nan,
        auc_hedge_posterior_for_labeler_uncertain=C.auc(unc, post[:, 2]),
        auc_omission_posterior_for_labeler_positive_unmentioned=C.auc(
            ((df[target] == 1) & (df.view1_mention == 0)).values, post[:, 1]),
        frac_conf_gt_099=float((resp.max(1) > 0.99).mean()),
        mix_by_quality_tercile=mix.round(3).to_dict(),
        corr_faithful_posterior_quality=float(np.corrcoef(post[:, 0], df.q_meas)[0, 1]),
    )
    return res, named, post


def analyse(st, fnd, target, out, prefix, title, load_images=None, seeds="0,1,2", n_cases=4000,
            radgraph_features=None, cnn_epochs=8, n_restarts=6, pairs_basename=True, notes=None):
    """Steps (a)-(d) on any study table with the columns subject_id, study_id, ViewPosition, order, txt,
    reader_id (may be NaN) and CheXpert-style finding columns (1 / 0 / -1 / NaN).
    load_images(samp) must return samp restricted to studies with an image, with img, q_meas, img_path."""
    import harness_current as H
    os.makedirs(out, exist_ok=True)
    res = {"n_studies_with_report": len(st), "n_patients": int(st.subject_id.nunique()), "target": target}
    if notes:
        res["notes"] = notes
    print("(a) concomitant link, all studies, labels only")
    per, pool = concomitant(st, fnd)
    res["concomitant_pooled"] = pool
    print("(b) current-prior axis")
    lon = longitudinal(st, fnd)
    em = []
    if load_images is not None:
        rng = np.random.default_rng(0)
        prank = dict(zip(st.subject_id.unique(), rng.permutation(st.subject_id.nunique())))
        samp = st.assign(_r=st.subject_id.map(prank)).sort_values(["_r", "order"]).drop(columns="_r")
        samp = samp.head(n_cases * 2)                 # whole patients, random order
        samp = load_images(samp).head(n_cases)
        if len(samp) >= 100:
            print("(a') concomitant link with image sharpness, image sample")
            per_q, pool_q = concomitant(samp, fnd, quality=samp.q_meas.values)
            res["concomitant_pooled_image_sample"] = pool_q
            per_q.to_csv(os.path.join(out, f"{prefix}_concomitant_with_sharpness.csv"), index=False)
            print("(c) latent-class EM on real views")
            for s in [int(x) for x in str(seeds).split(",")]:
                df, feat = em_views(samp, target, s, radgraph_features, cnn_epochs, H)
                r, named, post = em_face_validity(df, feat, target, s, H, n_restarts)
                r["seed"] = s; em.append(r)
                print(f"  seed {s}: shares {r['class_share']}; P(-1|hedge)={r['p_uncertain_given_named_hedge']:.3f} "
                      f"vs {r['p_uncertain_given_other']:.3f}; AUC {r['auc_hedge_posterior_for_labeler_uncertain']:.3f}")
            res["em"] = em
            pairs = samp[["img_path", "subject_id", "txt"]].rename(columns={"img_path": "filename", "subject_id": "patientid"})
            if pairs_basename:
                pairs["filename"] = pairs.filename.map(os.path.basename)  # runner finds images by basename under --img-dir
            pairs["y_target"] = (samp[target] == 1).astype(int).values
            pairs.to_csv(os.path.join(out, f"{prefix}_pairs.csv"), index=False)
            print(f"(d) wrote {os.path.join(out, prefix + '_pairs.csv')} for the Part C1 architecture runner (--mode pairs)")
        else:
            print("fewer than 100 studies with images: (a') and (c) skipped")
    per.to_csv(os.path.join(out, f"{prefix}_concomitant.csv"), index=False)
    lon.to_csv(os.path.join(out, f"{prefix}_longitudinal.csv"), index=False)
    C.save_json(res, os.path.join(out, f"{prefix}_results.json"))

    L = [f"# {title} ({res['n_studies_with_report']} frontal studies, {res['n_patients']} patients)", ""]
    if notes:
        L += [f"> {n}" for n in notes] + [""]
    L += [
         "**(a) Is 'uncertain' more frequent on AP (portable) films?** Odds ratio, uncertain vs definite label, "
         "among studies that mention the finding.", ""]
    if pool:
        L += [f"* pooled with finding fixed effects: OR {pool['odds_ratio']:.2f} "
              f"[{pool['ci_low']:.2f}, {pool['ci_high']:.2f}], p = {pool['p']:.2g}; "
              f"uncertain rate AP {pool['rate_AP']:.3f} vs PA {pool['rate_PA']:.3f}; n = {pool['n']}", ""]
    if len(per):
        L += C.md_table(per[["finding", "n_mentioned", "uncertain_rate", "odds_ratio", "ci_low", "ci_high", "p"]])
    L += ["", "**(b) Current-prior axis** (consecutive frontal studies of the same patient)", ""]
    L += C.md_table(lon)
    if em:
        E = pd.DataFrame([{**{"seed": r["seed"]}, **{f"share_{k}": v for k, v in r["class_share"].items()},
                           "P(-1 | hedge)": r["p_uncertain_given_named_hedge"], "P(-1 | other)": r["p_uncertain_given_other"],
                           "AUC hedge post.": r["auc_hedge_posterior_for_labeler_uncertain"],
                           "conf>0.99": r["frac_conf_gt_099"], "r(faithful, quality)": r["corr_faithful_posterior_quality"]}
                          for r in em])
        L += ["", f"**(c) Latent-class EM on real views, target = {target}** (face validity only; "
              f"features: {', '.join(em[0]['features'])})", ""]
        L += C.md_table(E)
    C.save_md(L, os.path.join(out, f"{prefix}_summary.md"))
    return res


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--jpg-dir", required=True)
    ap.add_argument("--reports-dir", default=None, help="MIMIC-CXR reports root (default: --jpg-dir)")
    ap.add_argument("--provider-csv", default=None)
    ap.add_argument("--radgraph-features", default=None)
    ap.add_argument("--target", default="Pleural Effusion")
    ap.add_argument("--n-cases", type=int, default=4000, help="studies sampled for the image-based EM (c)")
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--cnn-epochs", type=int, default=8)
    ap.add_argument("--n-restarts", type=int, default=6)
    ap.add_argument("--no-images", action="store_true")
    ap.add_argument("--out", default="results")
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    st, fnd = load_tables(a.jpg_dir)
    st = attach_reports(st, a.reports_dir or a.jpg_dir)
    if a.target not in fnd:
        sys.exit(f"target {a.target} not among {fnd}")
    st = attach_provider(st, a.provider_csv)
    loader = None if a.no_images else (lambda samp: attach_images(samp, a.jpg_dir))
    return analyse(st, fnd, a.target, a.out, "mimic", "D2 MIMIC-CXR", loader, a.seeds, a.n_cases,
                   a.radgraph_features, a.cnn_epochs, a.n_restarts)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile realdata/vindr_run.py
"""
D3 - VinDr-CXR (Nguyen et al., 2022; PhysioNet "vindr-cxr" 1.0.0, credentialed).

Role in Table VII: the reader-reader axis. Each training image was labelled independently by
three of 17 radiologists (rad_id R1..R17), so reader disagreement is observed directly. The
labels are structured findings, not free text, so the four-cause taxonomy cannot be scored;
what is checked is the reader side of the model:

  (a) How much do readers disagree? Per finding: prevalence, non-unanimous rate, Fleiss kappa.
  (b) Are reader traits stable? View 3 of the EM assumes a reader's tendency is a fixed trait.
      Dawid-Skene EM per finding gives each reader's sensitivity and specificity; a split-half
      test (each reader's images split at random into two halves) gives the correlation of a
      reader's positive-call rates across halves, over findings. A trait that is real should
      replicate across halves.
  (c) Do readers agree on where? For local labels called by >= 2 readers on the same image,
      the mean best-match IoU of their boxes.
  (d) Optional, needs the DICOM images: can a cross-fitted image CNN (the input side of the
      anticipation head) predict which images readers will disagree on, compared with two
      reference predictors that need no image model (number of findings the Dawid-Skene
      posterior calls present; image sharpness)?

FILES under --vindr-dir: annotations/image_labels_train.csv (image_id, rad_ID, 28 label columns),
annotations/annotations_train.csv (image_id, rad_id, class_name, x_min, y_min, x_max, y_max);
images train/<image_id>.dicom (optional). The Kaggle competition copy (train.csv with image_id,
class_name, rad_id, boxes) is accepted for (a)-(c) when image_labels_train.csv is absent.

USAGE  python3 vindr_run.py --vindr-dir <dir> [--with-images] [--max-images 3000] [--seeds 0,1,2]
"""
import argparse
import os
import sys
from itertools import combinations

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C


def load_labels(root):
    print("VinDr-CXR files:")
    f = C.find_file(root, "image_labels_train.csv")
    boxes_f = C.find_file(root, "annotations_train.csv") or C.find_file(root, "train.csv")
    boxes = C.read_table(boxes_f) if boxes_f else None
    if f:
        lab = C.read_table(f)
        idc, rc = C.col(lab, "image_id"), C.col(lab, "rad_id", "rad_ID")
        finds = [c for c in lab.columns if c not in (idc, rc)]
        lab = lab.rename(columns={idc: "image_id", rc: "rad_id"})
        lab[finds] = (lab[finds].fillna(0).astype(float) > 0).astype(int)
    elif boxes is not None:
        print("  image_labels_train.csv not found: image-level labels derived from box annotations")
        b = boxes.rename(columns={C.col(boxes, "image_id"): "image_id", C.col(boxes, "rad_id", "rad_ID"): "rad_id",
                                  C.col(boxes, "class_name"): "class_name"})
        lab = pd.crosstab([b.image_id, b.rad_id], b.class_name).clip(upper=1).reset_index()
        finds = [c for c in lab.columns if c not in ("image_id", "rad_id")]
    else:
        sys.exit("neither image_labels_train.csv nor annotations_train.csv/train.csv found")
    finds = [f for f in finds if f.strip().lower() != "no finding"]   # derived label, not a finding
    nr = lab.groupby("image_id").rad_id.nunique()
    print(f"  images {lab.image_id.nunique()}, readers {lab.rad_id.nunique()}, reads per image: "
          f"{nr.value_counts().sort_index().to_dict()}")
    return lab, finds, boxes


def agreement(lab, finds):
    rows = []
    g = lab.groupby("image_id")
    for f in finds:
        pos = g[f].sum(); n = g[f].count()
        k = C.fleiss_kappa(np.column_stack([pos, n - pos]))
        rows.append(dict(finding=f, prevalence_any_reader=float((pos > 0).mean()),
                         prevalence_majority=float((pos > n / 2).mean()),
                         non_unanimous=float(((pos > 0) & (pos < n)).mean()), fleiss_kappa=k))
    return pd.DataFrame(rows).sort_values("prevalence_any_reader", ascending=False)


def reader_traits(lab, finds, min_reads=200, seed=0):
    readers = sorted(lab.rad_id.unique())
    ri = {r: i for i, r in enumerate(readers)}
    imgs = sorted(lab.image_id.unique()); ii = {m: i for i, m in enumerate(imgs)}
    sens, spec = {}, {}
    disagree = {}
    for f in finds:
        L = -np.ones((len(imgs), len(readers)), dtype=int)
        L[lab.image_id.map(ii).values, lab.rad_id.map(ri).values] = lab[f].values
        if L.max() < 1 or (L == 1).sum() < 20:
            continue
        T, prior, conf = C.dawid_skene(L)
        sens[f] = conf[:, 1, 1]; spec[f] = conf[:, 0, 0]
        disagree[f] = T[:, 1]
    nreads = lab.groupby("rad_id").size().reindex(readers).values
    keep = nreads >= min_reads
    S = pd.DataFrame(sens, index=readers)[keep]; P = pd.DataFrame(spec, index=readers)[keep]
    # split-half stability of positive-call rates
    rng = np.random.default_rng(seed)
    lab = lab.copy(); lab["half"] = rng.integers(0, 2, len(lab))
    rates = lab.groupby(["rad_id", "half"])[finds].mean()
    rs = []
    for f in finds:
        r = rates[f].unstack("half").dropna()
        r = r[r.index.isin(S.index)]
        if len(r) >= 4 and r.std().min() > 0:
            rs.append(np.corrcoef(r[0], r[1])[0, 1])
    res = dict(n_readers_kept=int(keep.sum()), min_reads=min_reads,
               split_half_r_median=float(np.median(rs)) if rs else np.nan, split_half_n_findings=len(rs),
               sens_range_median_over_findings=float((S.max() - S.min()).median()) if len(S) else np.nan,
               spec_range_median_over_findings=float((P.max() - P.min()).median()) if len(P) else np.nan)
    post = pd.DataFrame(disagree, index=imgs)
    return res, S, P, post


def box_iou(boxes):
    if boxes is None:
        return pd.DataFrame()
    b = boxes.rename(columns={C.col(boxes, "rad_id", "rad_ID"): "rad_id"}).dropna(subset=["x_min"])
    b = b[b.class_name != "No finding"]

    def iou(p, q):
        ix = max(0, min(p[2], q[2]) - max(p[0], q[0])); iy = max(0, min(p[3], q[3]) - max(p[1], q[1]))
        inter = ix * iy
        u = (p[2] - p[0]) * (p[3] - p[1]) + (q[2] - q[0]) * (q[3] - q[1]) - inter
        return inter / u if u > 0 else 0.0
    rows = []
    for (img, cls), g in b.groupby(["image_id", "class_name"]):
        per = {r: g2[["x_min", "y_min", "x_max", "y_max"]].values for r, g2 in g.groupby("rad_id")}
        if len(per) < 2:
            continue
        vals = [np.mean([max(iou(p, q) for q in per[r2]) for p in per[r1]]) for r1, r2 in combinations(per, 2)]
        rows.append((cls, float(np.mean(vals))))
    d = pd.DataFrame(rows, columns=["class_name", "iou"])
    return d.groupby("class_name").iou.agg(["count", "mean", "median"]).reset_index().sort_values("count", ascending=False)


def image_probe(lab, finds, post, root, seeds, max_images, epochs, harness):
    idx = C.index_images(root, exts=(".dicom", ".dcm", ".png", ".jpg"))
    g = lab.groupby("image_id")
    pos, n = g[finds].sum(), g[finds].count()
    y = ((pos > 0) & (pos < n)).any(axis=1).astype(int)
    ids = [m for m in y.index if m in idx][:max_images]
    print(f"  images found: {len(ids)}")
    if len(ids) < 100:
        print("  fewer than 100 images: (d) skipped"); return None
    imgs = np.stack([C.load_gray(idx[m], 32) for m in ids]); yy = y.loc[ids].values.astype(np.float32)
    q = np.array([C.laplacian_var(x) for x in imgs])
    ds_count = (post.loc[ids] > 0.5).sum(axis=1).values
    from sklearn.model_selection import StratifiedKFold
    out = []
    for s in seeds:
        sc = np.zeros(len(ids), dtype=np.float32)
        for i, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=s).split(imgs, yy)):
            sc[te] = harness._score_cnn(harness._train_cnn(imgs[tr], yy[tr], epochs, s * 100 + i), imgs[te])
        out.append(C.auc(yy, sc))
    return dict(n=len(ids), disagree_rate=float(yy.mean()), auc_image_cnn_mean=float(np.mean(out)),
                auc_image_cnn_sd=float(np.std(out)), auc_ds_positive_count=C.auc(yy, ds_count),
                auc_sharpness=max(C.auc(yy, q), C.auc(yy, -q)), seeds=list(seeds))


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--vindr-dir", required=True)
    ap.add_argument("--with-images", action="store_true")
    ap.add_argument("--max-images", type=int, default=3000)
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--min-reads", type=int, default=200)
    ap.add_argument("--out", default="results")
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    lab, finds, boxes = load_labels(a.vindr_dir)
    A = agreement(lab, finds)
    T, S, P, post = reader_traits(lab, finds, a.min_reads)
    B = box_iou(boxes)
    res = dict(n_images=int(lab.image_id.nunique()), n_readers=int(lab.rad_id.nunique()), reader_traits=T)
    if a.with_images:
        import harness_current as H
        res["image_probe"] = image_probe(lab, finds, post, a.vindr_dir, [int(s) for s in a.seeds.split(",")],
                                         a.max_images, a.epochs, H)
    A.to_csv(os.path.join(a.out, "vindr_agreement.csv"), index=False)
    S.to_csv(os.path.join(a.out, "vindr_reader_sensitivity.csv")); P.to_csv(os.path.join(a.out, "vindr_reader_specificity.csv"))
    C.save_json(res, os.path.join(a.out, "vindr_results.json"))
    L = [f"# D3 VinDr-CXR: reader-reader axis ({res['n_images']} images, {res['n_readers']} readers)", "",
         "**(a) Agreement per finding** (three independent reads per image)", ""]
    L += C.md_table(A)
    L += ["", "**(b) Are reader traits stable?**", "",
          f"* readers with >= {T['min_reads']} reads: {T['n_readers_kept']}",
          f"* split-half correlation of per-reader positive-call rate, median over {T['split_half_n_findings']} findings: "
          f"{T['split_half_r_median']:.3f}",
          f"* range of per-reader sensitivity (Dawid-Skene), median over findings: {T['sens_range_median_over_findings']:.3f}; "
          f"specificity: {T['spec_range_median_over_findings']:.3f}"]
    if len(B):
        L += ["", "**(c) Localisation agreement** (mean best-match box IoU between readers who both called the finding)", ""]
        L += C.md_table(B)
    if res.get("image_probe"):
        p = res["image_probe"]
        L += ["", "**(d) Does the image predict reader disagreement?** (AUC, 5-fold cross-fitted)", "",
              f"* n = {p['n']}, disagreement rate {p['disagree_rate']:.3f}",
              f"* image CNN {p['auc_image_cnn_mean']:.3f} +/- {p['auc_image_cnn_sd']:.3f}; "
              f"reference: Dawid-Skene positive count {p['auc_ds_positive_count']:.3f}; sharpness {p['auc_sharpness']:.3f}"]
    C.save_md(L, os.path.join(a.out, "vindr_summary.md"))
    return res


if __name__ == "__main__":
    main()

---
# Part E — Open and easier-access alternatives to the Table VII datasets

Part D needs PhysioNet credentialing for five of its six datasets. Part E runs the same checks on datasets you can obtain without it, so the real-data section of the paper does not have to wait for credentialing. REFLACX is the only check with no open substitute, because every chest X-ray gaze dataset (EGD-CXR, GazeSearch) is itself credentialed. The second-institution role of CheXpert needs no substitute: CheXpert itself requires only Stanford AIMI registration (Part D5), and ReXGradient already spans three US health systems.

| Step | Dataset | Replaces (Table VII role) | Access | Runs |
|---|---|---|---|---|
| E0 | CheXbert + RadGraph models (open weights) | MIMIC-CXR's CheXpert labels and RadGraph inference | `pip install f1chexbert radgraph`; weights from Hugging Face on first use | labels reports that come without labels; falls back to a rule-based labeler |
| E1 | ReXGradient-160K | MIMIC-CXR: image–report and current–prior axes | Hugging Face licence click-through | the D2 analysis: concomitant link, current–prior axis, EM on real views, architecture pairs (E1b) |
| E2 | NIH ChestX-ray14 + Google expert labels | VinDr-CXR: reader–reader axis | Google request form (same files as Part C2) | the D3 analysis, plus whether *hedging* is a stable reader trait (the View 3 assumption) |
| E3 | Errors injected into IU or ReXGradient reports | CorBenchX: synthetic errors for triage | none beyond the source reports | the D6 analysis on rule-injected omission, insertion, side confusion, spelling, other and hedge errors |
| E4 | — | — | — | status of every Part D and Part E step |

**Two caveats.**

* **Model labels, not expert labels.** CheXbert and RadGraph are models, so E0 labels are machine labels; MIMIC-CXR's own CheXpert labels are machine labels too. If CheXbert cannot be installed or downloaded, E0 falls back to the rule-based labeler and says so. In that case the EM face-validity check (c) is circular, because the "uncertain" label and View 1 come from the same rules, and the summary states this. Checks (a) and (b) remain valid.
* **Injected errors are not an independent benchmark.** E3's errors are made by rules written for this notebook, so they test the text view and Algorithm 2 under controlled corruption only. Report them as such.

**Testing status.** As in Part D, every runner was tested end to end on mock files built to the documented layout. Expect an adjustment on the first real run. ReXGradient's metadata columns are not listed on its public dataset card, so they are discovered at run time and can be set with `--col-*` options; the runner prints what it found.

## E0. Labels for reports that come without them: CheXbert, RadGraph, or the rule-based fallback

In [ ]:
%%writefile realdata/label_reports.py
"""
E0 - Label free-text reports the way MIMIC-CXR ships them, for datasets that come without labels
(ReXGradient-160K, IU/Open-I).

MIMIC-CXR comes with CheXpert-labeler outputs (1 / 0 / -1 uncertain / blank) and RadGraph
inference. The open alternatives have reports only, so this step produces the same two things:

  * CheXpert-style labels for 14 findings, from the open CheXbert model (pip install f1chexbert;
    weights are downloaded from Hugging Face on first use) or, if it is not available, from the
    rule-based labeler in rd_common.lexical_labels.
  * View 1 features (rg_hedge, rg_mention, rg_negated) from the open RadGraph model
    (pip install radgraph; RadGraph or RadGraph-XL weights from Hugging Face), if available.

Which labeler produced each column is written to the output, and the summary says so. CheXbert
and RadGraph are models, not expert annotations: results built on them are face-validity checks.
When both CheXbert and the rule labeler run, their agreement per finding is reported.

INPUT  a CSV with a study id column and a report text column (names given with --id-col/--text-col)
OUTPUT <out>/<prefix>_labels.csv : study id, 14 finding columns, label_source
       <out>/<prefix>_radgraph_v1.csv : study_id, rg_hedge, rg_mention, rg_negated (if RadGraph ran)

USAGE  python3 label_reports.py --reports reports.csv --id-col study_id --text-col txt --prefix rexgradient
         [--labeler auto|chexbert|lexical] [--radgraph auto|on|off] [--max-reports 20000]
"""
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C


def chexbert_labeler():
    try:
        from f1chexbert import F1CheXbert
        m = F1CheXbert(device="cpu")
    except Exception as e:                       # package missing, or weights not downloadable
        print(f"  CheXbert not available ({type(e).__name__}: {str(e)[:120]}); "
              "install with `pip install f1chexbert` on a machine that can reach huggingface.co")
        return None
    names = m.target_names

    def lab(text):
        v = m.get_label(str(text)[:4000], mode="classification")
        return {n: (np.nan if x == "" else float(x)) for n, x in zip(names, v)}
    return lab


def radgraph_model(kind):
    try:
        from radgraph import RadGraph
        return RadGraph(model_type=kind)
    except Exception as e:
        print(f"  RadGraph not available ({type(e).__name__}: {str(e)[:120]}); "
              "install with `pip install radgraph` on a machine that can reach huggingface.co")
        return None


def radgraph_features(rg, texts, batch=16):
    rows = []
    for i in range(0, len(texts), batch):
        out = rg(list(texts[i:i + batch]))
        for k in sorted(out, key=lambda x: int(x)):
            ents = out[k].get("entities", {})
            st = [C.radgraph_status(e.get("label")) for e in ents.values()
                  if any(t in str(e.get("tokens", "")).lower() for t in C.ANY_FINDING)]
            rows.append((float("OBS-U" in st), float("OBS-DP" in st), float("OBS-DA" in st)))
    return np.array(rows) if rows else np.zeros((0, 3))


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--reports", required=True)
    ap.add_argument("--id-col", default="study_id")
    ap.add_argument("--text-col", default="txt")
    ap.add_argument("--prefix", required=True)
    ap.add_argument("--labeler", choices=["auto", "chexbert", "lexical"], default="auto")
    ap.add_argument("--radgraph", choices=["auto", "on", "off"], default="auto")
    ap.add_argument("--radgraph-model", default="radgraph-xl")
    ap.add_argument("--max-reports", type=int, default=20000, help="model labelers are slow on CPU (0 = all)")
    ap.add_argument("--out", default="results")
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    d = pd.read_csv(a.reports)
    d = d[[a.id_col, a.text_col]].dropna().drop_duplicates(a.id_col).rename(columns={a.id_col: "study_id", a.text_col: "txt"})
    print(f"E0 labelling {len(d)} reports")
    t0 = time.time()
    lex = pd.DataFrame([C.lexical_labels(t) for t in d.txt], index=d.index)
    lab, src = lex.copy(), "rule-based lexical labeler"
    res = {"n": len(d)}
    if a.labeler in ("auto", "chexbert"):
        cb = chexbert_labeler()
        if cb is None and a.labeler == "chexbert":
            sys.exit("CheXbert requested but not available")
        if cb is not None:
            sub = d if not a.max_reports else d.head(a.max_reports)
            print(f"  CheXbert on {len(sub)} reports ...")
            cbl = pd.DataFrame([cb(t) for t in sub.txt], index=sub.index)
            lab = lab.astype(float)
            lab.loc[sub.index, cbl.columns] = cbl.values
            src = "CheXbert" if len(sub) == len(d) else f"CheXbert ({len(sub)} reports), rule-based for the rest"
            agree = {}
            for f in cbl.columns:
                if f in lex.columns:
                    x = cbl[f].fillna(9).values; y = lex.loc[sub.index, f].fillna(9).values
                    agree[f] = dict(exact=float((x == y).mean()), kappa=C.cohen_kappa(x, y),
                                    uncertain_rate_chexbert=float((x == -1).mean()), uncertain_rate_rule=float((y == -1).mean()))
            res["chexbert_vs_rule"] = agree
    lab["label_source"] = src
    out = pd.concat([d[["study_id"]], lab], axis=1)
    out.to_csv(os.path.join(a.out, f"{a.prefix}_labels.csv"), index=False)
    res["label_source"] = src
    res["uncertain_rate"] = {f: float((lab[f] == -1).mean()) for f in C.CHEXPERT14 if f in lab}
    if a.radgraph != "off":
        rg = radgraph_model(a.radgraph_model)
        if rg is None and a.radgraph == "on":
            sys.exit("RadGraph requested but not available")
        if rg is not None:
            sub = d if not a.max_reports else d.head(a.max_reports)
            print(f"  RadGraph ({a.radgraph_model}) on {len(sub)} reports ...")
            f = radgraph_features(rg, sub.txt.tolist())
            pd.DataFrame({"study_id": sub.study_id.values, "rg_hedge": f[:, 0], "rg_mention": f[:, 1],
                          "rg_negated": f[:, 2]}).to_csv(os.path.join(a.out, f"{a.prefix}_radgraph_v1.csv"), index=False)
            res["radgraph_reports"] = len(sub)
    res["seconds"] = round(time.time() - t0, 1)
    C.save_json(res, os.path.join(a.out, f"{a.prefix}_labels_info.json"))
    print(f"  labels from: {src}; {res['seconds']} s")
    return res


if __name__ == "__main__":
    main()

## E1. ReXGradient-160K in place of MIMIC-CXR

After accepting the licence on Hugging Face, download the repository and join the image parts: `cat deid_png.part* > deid_png.tar && tar -xf deid_png.tar`. Set `REXGRADIENT_DIR` to the repository folder and `REXGRADIENT_IMAGES_DIR` to the extracted images. `REX_MAX_LABEL` caps how many studies are labelled, whole patients at a time; CheXbert labels about 10–20 reports per second on a CPU.

In [ ]:
%%writefile realdata/rexgradient_run.py
"""
E1 - ReXGradient-160K (Zhang et al., 2025, arXiv:2505.00228): 160,000 chest X-ray studies, 273,004
images, 109,487 patients from three US health systems, with Indication / Comparison / Findings /
Impression sections and several studies per patient. Access: sign the licence on Hugging Face
(huggingface.co/datasets/rajpurkarlab/ReXGradient-160K), no PhysioNet credentialing. Images are
distributed as deid_png.part00..09 (cat deid_png.part* > deid_png.tar; tar -xf deid_png.tar).

Open replacement for MIMIC-CXR in Table VII (image-report and current-prior axes). It runs the
same analysis as D2 (mimic_run.analyse): (a) concomitant link AP vs PA, (b) current-prior axis,
(c) the EM of Part A on real views, (d) pairs for the Section II architecture. Labels come from
E0 (CheXbert or the rule-based labeler) because ReXGradient ships none; there is no reader id,
so View 3 is omitted.

SCHEMA. The public dataset card does not list the metadata columns, so they are discovered: every
CSV / JSON / JSONL under --rex-dir is read and the first table with a patient column and a
findings or impression column is used. Candidates (case-insensitive), overridable with --col-*:
  patient : PatientID, patient_id        study : AccessionNumber, StudyInstanceUid, study_id, id
  date    : StudyDate, study_date        view  : ViewPosition, view_position, view
  findings: Findings   impression: Impression   images: ImagePath(s), image_path(s), images
If the view is not a column, a JSON whose name contains "view_position" mapping image -> view is used.

USAGE  python3 rexgradient_run.py --rex-dir <repo> --images-dir <extracted deid_png> [--no-images]
         [--target "Pleural Effusion"] [--labeler auto|chexbert|lexical] [--max-label 20000]
"""
import argparse
import ast
import json
import os
import re
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C
import mimic_run as MM
import label_reports as LR

CANDS = dict(patient=["PatientID", "patient_id", "patientid", "subject_id"],
             study=["AccessionNumber", "StudyInstanceUid", "StudyInstanceUID", "study_id", "studyid", "id"],
             date=["StudyDate", "study_date", "date"],
             view=["ViewPosition", "view_position", "view", "Projection"],
             findings=["Findings", "findings", "section_findings"],
             impression=["Impression", "impression", "section_impression"],
             images=["ImagePath", "ImagePaths", "image_path", "image_paths", "images", "ImageFile", "png_path"])


def _read(p):
    if p.endswith(".jsonl"):
        return pd.read_json(p, lines=True)
    if p.endswith(".json"):
        d = json.load(open(p))
        if isinstance(d, dict):
            vals = list(d.values())
            if vals and all(isinstance(v, dict) for v in vals):
                return pd.DataFrame([dict(v, _key=k) for k, v in d.items()])
            return pd.DataFrame({"_key": list(d.keys()), "_value": vals})
        return pd.json_normalize(d)
    return pd.read_csv(p)


def discover(root, over):
    files = [p for p in C.find_files(root, r"\.(csv|json|jsonl)$") if "bbox" not in os.path.basename(p).lower()]
    tables, view_map = [], None
    for p in files:
        try:
            t = _read(p)
        except Exception as e:
            print(f"  could not read {p}: {e}"); continue
        print(f"  {os.path.relpath(p, root)}: {len(t)} rows; columns = {list(t.columns)[:20]}")
        if "view_position" in os.path.basename(p).lower():
            view_map = t
        cols = {k: (over.get(k) or C.col(t, *v, required=False)) for k, v in CANDS.items()}
        if cols["patient"] and (cols["findings"] or cols["impression"]):
            t = t.rename(columns={v: k for k, v in cols.items() if v})
            t["_file"] = os.path.basename(p)
            tables.append(t)
    if not tables:
        sys.exit("no metadata table with a patient column and a Findings/Impression column was found; "
                 "pass --col-patient / --col-findings / --col-impression with the names printed above")
    d = pd.concat(tables, ignore_index=True)
    print(f"  using {len(tables)} table(s), {len(d)} rows; mapped columns: "
          f"{[k for k in CANDS if k in d.columns]}")
    return d, view_map


def _first_image(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    if isinstance(v, str) and v.strip().startswith("["):
        try:
            return list(ast.literal_eval(v))
        except Exception:
            pass
    return [v] if isinstance(v, str) else []


def norm_view(v):
    v = str(v).upper().replace("-", "").replace(" ", "")
    if v.startswith("PA") or v == "POSTEROANTERIOR":
        return "PA"
    if v.startswith("AP") or "ANTEROPOSTERIOR" in v or "SUPINE" in v or "PORTABLE" in v:
        return "AP"
    return None


def build_table(d, view_map):
    if "study" not in d.columns:
        d["study"] = np.arange(len(d))
        print("  no study column found: one row = one study")
    if "images" in d.columns:
        d["image_list"] = d["images"].map(_first_image)
    else:
        d["image_list"] = [[] for _ in range(len(d))]
    if "view" not in d.columns and view_map is not None:
        vm = view_map
        if {"_key", "_value"} <= set(vm.columns):
            m = dict(zip(vm._key.astype(str), vm._value))
        else:
            kc = C.col(vm, "_key", "image_path", "ImagePath", "path", required=False)
            vc = C.col(vm, "ViewPosition", "view_position", "view", required=False)
            m = dict(zip(vm[kc].astype(str), vm[vc])) if kc and vc else {}
        bn = {os.path.basename(k): v for k, v in m.items()}
        d["view"] = [next((m.get(str(i), bn.get(os.path.basename(str(i)))) for i in L
                           if (m.get(str(i)) or bn.get(os.path.basename(str(i))))), None) for L in d.image_list]
        print(f"  views taken from the view-position file for {d.view.notna().sum()} studies")
    txt = (d.get("findings", pd.Series("", index=d.index)).fillna("").astype(str) + " " +
           d.get("impression", pd.Series("", index=d.index)).fillna("").astype(str)).str.strip()
    st = pd.DataFrame(dict(subject_id=d["patient"].astype(str), study_id=d["study"].astype(str), txt=txt,
                           ViewPosition=d["view"].map(norm_view) if "view" in d.columns else None,
                           order=pd.to_datetime(d["date"], errors="coerce").astype("int64") if "date" in d.columns else np.arange(len(d)),
                           img_rel=[L[0] if L else None for L in d.image_list]))
    st = st[st.txt.str.len() > 0].drop_duplicates("study_id")
    st["reader_id"] = np.nan
    print(f"  studies {len(st)}, patients {st.subject_id.nunique()}, views {st.ViewPosition.value_counts(dropna=False).to_dict()}")
    return st


def image_loader(images_dir):
    idx = None

    def load(samp):
        nonlocal idx
        paths = []
        for r in samp.img_rel:
            p = None
            if isinstance(r, str):
                cand = r if os.path.isabs(r) else os.path.join(images_dir, r)
                if os.path.exists(cand):
                    p = cand
                else:
                    if idx is None:
                        idx = C.index_images(images_dir, exts=(".png", ".jpg"))
                    p = idx.get(os.path.splitext(os.path.basename(r))[0])
            paths.append(p)
        s = samp.assign(img_path=paths)
        s = s[s.img_path.notna()].copy()
        print(f"  images found for {len(s)} / {len(samp)} sampled studies")
        s["img"] = [C.load_gray(p, 32) for p in s.img_path]
        s["q_meas"] = s.img.map(C.laplacian_var)
        return s
    return load


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--rex-dir", required=True)
    ap.add_argument("--images-dir", default=None)
    ap.add_argument("--no-images", action="store_true")
    ap.add_argument("--target", default="Pleural Effusion")
    ap.add_argument("--labeler", default="auto")
    ap.add_argument("--radgraph", default="auto")
    ap.add_argument("--max-label", type=int, default=20000)
    ap.add_argument("--n-cases", type=int, default=4000)
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--out", default="results")
    for k in CANDS:
        ap.add_argument(f"--col-{k}", default=None)
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    print("ReXGradient-160K metadata:")
    d, vm = discover(a.rex_dir, {k: getattr(a, f"col_{k}") for k in CANDS})
    st = build_table(d, vm)
    st = st[st.ViewPosition.isin(["AP", "PA"])] if st.ViewPosition.notna().any() else st.assign(ViewPosition="PA")
    if st.ViewPosition.nunique() < 2:
        print("  WARNING: no AP/PA information: the concomitant test (a) is uninformative")
    rng = np.random.default_rng(0)                    # whole patients, random order, for the labeler budget
    prank = dict(zip(st.subject_id.unique(), rng.permutation(st.subject_id.nunique())))
    st = st.assign(_r=st.subject_id.map(prank)).sort_values(["_r", "order"]).drop(columns="_r")
    if a.max_label:
        st = st.head(a.max_label)
    rep = os.path.join(a.out, "rexgradient_reports.csv")
    st[["study_id", "txt"]].to_csv(rep, index=False)
    LR.main(["--reports", rep, "--prefix", "rexgradient", "--labeler", a.labeler, "--radgraph", a.radgraph,
             "--max-reports", "0", "--out", a.out])
    lab = pd.read_csv(os.path.join(a.out, "rexgradient_labels.csv"), dtype={"study_id": str})
    fnd = [f for f in MM.FINDINGS if f in lab.columns]
    st = st.merge(lab, on="study_id", how="inner")
    rgf = os.path.join(a.out, "rexgradient_radgraph_v1.csv")
    loader = None if (a.no_images or not a.images_dir) else image_loader(a.images_dir)
    src = lab.label_source.iloc[0] if len(lab) else "?"
    notes = []
    if not str(src).startswith("CheXbert"):
        notes.append("Labels come from the rule-based labeler. Check (a) and (b) are valid, but check (c) is circular "
                     "unless View 1 comes from RadGraph: the hedge component and the 'uncertain' label are then read off "
                     "the same rules. Install f1chexbert (and radgraph) and re-run for an independent check.")
    return MM.analyse(st, fnd, a.target, a.out, "rexgradient", f"E1 ReXGradient-160K, labels: {src}", loader,
                      a.seeds, a.n_cases, rgf if os.path.exists(rgf) else None, pairs_basename=False, notes=notes)


if __name__ == "__main__":
    main()

In [ ]:
import rexgradient_run as RX
def _rex():
    args = ["--rex-dir", REXGRADIENT_DIR, "--target", MIMIC_TARGET, "--labeler", REX_LABELER,
            "--max-label", str(REX_MAX_LABEL), "--n-cases", str(MIMIC_N_CASES), "--out", RD_OUT,
            "--seeds", "0" if QUICK else "0,1,2"]
    args += ["--images-dir", REXGRADIENT_IMAGES_DIR] if REXGRADIENT_IMAGES_DIR else ["--no-images"]
    return RX.main(args)
run_d("E1", "ReXGradient-160K", REXGRADIENT_DIR, _rex)
show_md("rexgradient_summary.md")

In [ ]:
# E1b: the Section II architecture on ReXGradient image-report pairs (as D2b does for MIMIC)
def _arch_on_pairs(prefix, img_dir, label):
    import shutil, fixes_experiment as FX, iu_xray_run as IU
    pairs = os.path.join(RD_OUT, f"{prefix}_pairs.csv")
    if not os.path.exists(pairs):
        raise SystemExit(f"{prefix}_pairs.csv not found: the step before must run with images")
    res_dir = os.path.abspath("fullarch/results")
    keep = {f: os.path.join(res_dir, f) for f in ("iu_xray_results.json", "iu_xray_summary.md")}
    backup = {f: p + ".bak" for f, p in keep.items() if os.path.exists(p)}
    for f, b in backup.items():
        shutil.move(keep[f], b)                      # protect the C1 (IU) outputs
    old = FX.harness; FX.harness = harness_current
    try:
        with in_dir(os.path.abspath("fullarch")):
            sys.argv = ["iu_xray_run.py", img_dir, "--mode", "pairs", "--pairs-csv", pairs, "--img-dir", img_dir,
                        "--seeds", "0" if QUICK else "0,1,2", "--configs", "F5NR,F0C0,F3C0,F4C0,F4C1a", "--epochs", "30"]
            IU.main()
        shutil.move(keep["iu_xray_results.json"], os.path.join(RD_OUT, f"{prefix}_arch_results.json"))
        sm = os.path.join(RD_OUT, f"{prefix}_arch_summary.md")
        shutil.move(keep["iu_xray_summary.md"], sm)
        txt = open(sm).read(); open(sm, "w").write(txt.replace("IU X-ray run", label))
    finally:
        FX.harness = old
        for f, b in backup.items():
            shutil.move(b, keep[f])
if RUN_ALT_ARCH:
    run_d("E1b", "ReXGradient architecture", REXGRADIENT_IMAGES_DIR,
          lambda: _arch_on_pairs("rexgradient", REXGRADIENT_IMAGES_DIR, "ReXGradient-160K architecture run (E1b)"))
    show_md("rexgradient_arch_summary.md")

## E2. NIH ChestX-ray14 expert reads in place of VinDr-CXR

The same reader-level analysis as D3: Fleiss κ, Dawid–Skene reader sensitivity and specificity, and a split-half test. The NIH reads add an explicit HEDGE answer, which allows the assumption behind View 3 to be tested directly. If a reader's hedging is a trait, their hedge rate on one random half of their images should predict the rate on the other half. The image-side test (does the image predict disagreement) is Part C2. Uses `NIH_LABELS_DIR` from the configuration cell.

In [ ]:
%%writefile realdata/nih_traits_run.py
"""
E2 - Reader-reader axis without PhysioNet: NIH ChestX-ray14 with the Google expert labels
(Majkowska et al., Radiology 2019). Free request form on the Google Cloud NIH chest X-ray page;
the same files Part C2 uses (four_findings_expert_labels/individual_readers.csv, or the
all_findings_expert_labels set). Each image has independent reads by three radiologists
answering YES / NO / HEDGE per finding.

Open replacement for VinDr-CXR in Table VII. It runs the VinDr analysis (vindr_run) on these reads
and adds the one thing VinDr cannot give: an explicit HEDGE answer per reader. That allows View 3's
assumption to be tested directly: is a reader's hedging rate a stable trait? (split-half test of
each reader's hedge rate, and between-reader spread). Part C2 already tests the image side
(does the image predict disagreement); this step needs no images.

USAGE  python3 nih_traits_run.py --labels-dir <four_findings_expert_labels> [--min-reads 100]
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C
import vindr_run as VD

VALUES = {"YES", "NO", "HEDGE"}


def load(root):
    f = C.find_file(root, "individual_readers.csv") or C.find_file(root, "test_individual_readers.csv")
    if f is None:
        sys.exit("individual_readers.csv not found under " + root)
    print("NIH expert reads:")
    d = C.read_table(f)
    idc = next((c for c in d.columns if "image" in c.lower()), d.columns[0])
    rc = next((c for c in d.columns if "reader" in c.lower()), None)
    if rc is None:
        sys.exit("no reader column (name containing 'reader')")
    finds = [c for c in d.columns if c not in (idc, rc) and d[c].notna().any()
             and set(d[c].dropna().astype(str).str.upper().unique()) <= VALUES]
    print(f"  image col {idc}; reader col {rc}; findings {finds}")
    d = d.rename(columns={idc: "image_id", rc: "rad_id"})
    for c in finds:
        d[c] = d[c].astype(str).str.upper().where(d[c].notna(), None)
    return d, finds


def hedge_traits(d, finds, min_reads, seed=0):
    rng = np.random.default_rng(seed)
    h = d[["image_id", "rad_id"]].copy()
    h["hedge_any"] = (d[finds] == "HEDGE").any(axis=1).astype(int)
    for f in finds:
        h[f] = (d[f] == "HEDGE").astype(int)
    n = h.groupby("rad_id").size()
    keep = n[n >= min_reads].index
    h = h[h.rad_id.isin(keep)].copy()
    h["half"] = rng.integers(0, 2, len(h))
    r = h.groupby(["rad_id", "half"]).hedge_any.mean().unstack("half").dropna()
    rate = h.groupby("rad_id").hedge_any.mean()
    # does the reader explain hedging beyond the image? logistic with image fixed effects is too
    # large; use the share of hedges that come from images where another reader did not hedge
    g = h.groupby("image_id").hedge_any
    others = (g.transform("sum") - h.hedge_any) / (g.transform("count") - 1).clip(lower=1)
    lone = float(((h.hedge_any == 1) & (others == 0)).sum() / max(h.hedge_any.sum(), 1))
    return dict(n_readers_kept=int(len(keep)), min_reads=min_reads,
                hedge_rate_min=float(rate.min()) if len(rate) else np.nan,
                hedge_rate_median=float(rate.median()) if len(rate) else np.nan,
                hedge_rate_max=float(rate.max()) if len(rate) else np.nan,
                split_half_r=float(np.corrcoef(r[0], r[1])[0, 1]) if len(r) >= 3 else np.nan,
                share_of_hedges_where_no_other_reader_hedged=lone), rate


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--labels-dir", required=True)
    ap.add_argument("--min-reads", type=int, default=100)
    ap.add_argument("--out", default="results")
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    d, finds = load(a.labels_dir)
    pos = d[["image_id", "rad_id"]].copy()
    for f in finds:
        pos[f] = (d[f] == "YES").astype(int)
    A = VD.agreement(pos, finds)
    T, S, P, post = VD.reader_traits(pos, finds, a.min_reads)
    H, rate = hedge_traits(d, finds, a.min_reads)
    res = dict(n_images=int(d.image_id.nunique()), n_readers=int(d.rad_id.nunique()), reader_traits_positive=T,
               hedge_traits=H)
    A.to_csv(os.path.join(a.out, "nih_traits_agreement.csv"), index=False)
    rate.rename("hedge_rate").to_csv(os.path.join(a.out, "nih_reader_hedge_rates.csv"))
    C.save_json(res, os.path.join(a.out, "nih_traits_results.json"))
    L = [f"# E2 NIH ChestX-ray14 expert reads: reader-reader axis ({res['n_images']} images, {res['n_readers']} readers)", "",
         "**(a) Agreement per finding** (YES vs NO/HEDGE)", ""] + C.md_table(A)
    L += ["", "**(b) Are reader traits stable?** (positive calls, Dawid-Skene)", "",
          f"* readers with >= {T['min_reads']} reads: {T['n_readers_kept']}; split-half r of positive-call rate, "
          f"median over findings: {T['split_half_r_median']:.3f}; sensitivity range {T['sens_range_median_over_findings']:.3f}, "
          f"specificity range {T['spec_range_median_over_findings']:.3f}",
          "", "**(c) Is hedging a reader trait?** (the assumption behind View 3)", "",
          f"* per-reader rate of any HEDGE answer: min {H['hedge_rate_min']:.3f}, median {H['hedge_rate_median']:.3f}, "
          f"max {H['hedge_rate_max']:.3f} ({H['n_readers_kept']} readers)",
          f"* split-half correlation of the hedge rate: {H['split_half_r']:.3f}",
          f"* share of hedges on images where no other reader hedged: {H['share_of_hedges_where_no_other_reader_hedged']:.3f}"]
    C.save_md(L, os.path.join(a.out, "nih_traits_summary.md"))
    return res


if __name__ == "__main__":
    main()

In [ ]:
import nih_traits_run as NT
run_d("E2", "NIH expert reads (reader traits)", NIH_LABELS_DIR,
      lambda: NT.main(["--labels-dir", NIH_LABELS_DIR, "--out", RD_OUT]))
show_md("nih_traits_summary.md")

## E3. Injected errors in place of CorBenchX

Real reports from IU/Open-I (`IU_DATA_DIR`), or from ReXGradient if E1 ran, are corrupted by rules with the five CorBenchX error types plus a *hedge* type, which CorBenchX lacks. The pairs are then scored by the D6 runner. The injected errors are only as realistic as the rules that made them, so the result says what the text view and Algorithm 2 can and cannot see, not how often such errors occur.

In [ ]:
%%writefile realdata/inject_errors.py
"""
E3 - Injected report errors without PhysioNet (an open stand-in for CorBenchX / ReXErr).

Takes real reports from an open dataset (IU/Open-I indiana_reports.csv, the ReXGradient table
written by E1, or any CSV with an id and a text column) and writes clean/erroneous pairs with
the five CorBenchX error types, plus one type CorBenchX lacks:

  omission        drop a sentence that asserts a finding
  insertion       append a sentence asserting a finding the report does not mention
  side confusion  swap left and right
  spelling error  drop one letter from a finding word
  other           change a measurement unit (cm <-> mm) or, if none, merge two sentences
  hedge           (not in CorBenchX) turn an asserted finding into "possible ..."

The injection is rule-based and deterministic given the seed, so it is a controlled stress test of
the paper's text view and triage, not an independent benchmark: say so when reporting it. The
output has the columns D6 (corbenchx_run) detects automatically.

USAGE  python3 inject_errors.py --reports <csv> [--id-col uid --text-col txt] [--iu] [--n 5000]
"""
import argparse
import os
import re
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rd_common as C

INSERT = ["There is a small right pneumothorax.", "There is a moderate left pleural effusion.",
          "Right lower lobe consolidation is present.", "The heart is enlarged.",
          "There is a 1.5 cm nodule in the left upper lobe.", "Mild pulmonary edema is present."]


def asserted_sentences(text):
    out = []
    for i, s in enumerate(C.sentences(text)):
        low = s.lower()
        if any(t in low for t in C.ANY_FINDING) and not C.HEDGE_RX.search(low) and not C.NEG_RX.search(low):
            out.append(i)
    return out


def inject(text, kind, rng):
    S = C.sentences(text)
    A = asserted_sentences(text)
    if kind == "omission" and A:
        i = A[rng.integers(len(A))]
        return " ".join(s for j, s in enumerate(S) if j != i), f"removed: {S[i]}"
    if kind == "insertion":
        low = text.lower()
        cand = [s for s in INSERT if not any(t in low for t in C.ANY_FINDING if t in s.lower())]
        if cand:
            s = cand[rng.integers(len(cand))]
            return text.rstrip() + " " + s, f"inserted: {s}"
    if kind == "side confusion" and re.search(r"\b(left|right)\b", text, re.I):
        sw = {"left": "right", "right": "left", "Left": "Right", "Right": "Left", "LEFT": "RIGHT", "RIGHT": "LEFT"}
        return re.sub(r"\b(left|right|Left|Right|LEFT|RIGHT)\b", lambda m: sw[m.group(0)], text), "left/right swapped"
    if kind == "spelling error":
        words = [m for m in re.finditer(r"[A-Za-z]{6,}", text) if any(t in m.group(0).lower() for t in C.ANY_FINDING)]
        if words:
            m = words[rng.integers(len(words))]
            w = m.group(0); k = int(rng.integers(1, len(w) - 1))
            return text[:m.start()] + w[:k] + w[k + 1:] + text[m.end():], f"misspelled: {w}"
    if kind == "other":
        if re.search(r"\b\d+(\.\d+)?\s*cm\b", text):
            return re.sub(r"\b(\d+(?:\.\d+)?)\s*cm\b", r"\1 mm", text, count=1), "unit cm -> mm"
        if len(S) >= 2:
            return S[0].rstrip(".") + ", " + S[1][0].lower() + S[1][1:] + " " + " ".join(S[2:]), "sentences merged"
    if kind == "hedge" and A:
        i = A[rng.integers(len(A))]
        s = S[i].rstrip(".")
        S2 = list(S); S2[i] = "Possible " + s[0].lower() + s[1:] + "."
        return " ".join(S2), f"hedged: {S[i]}"
    return None, None


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--reports", required=True, help="CSV, or the IU folder with --iu")
    ap.add_argument("--iu", action="store_true", help="--reports is the IU/Open-I folder (indiana_reports.csv)")
    ap.add_argument("--id-col", default="study_id")
    ap.add_argument("--text-col", default="txt")
    ap.add_argument("--n", type=int, default=5000, help="clean reports used")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="results")
    a = ap.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    if a.iu:
        f = C.require(C.find_file(a.reports, "indiana_reports.csv"), "indiana_reports.csv")
        d = C.read_table(f)
        d.columns = [c.lower() for c in d.columns]
        clean = lambda s: re.sub(r"\s+", " ", re.sub(r"\bx{2,}\b", " ", "" if pd.isna(s) else str(s), flags=re.I)).strip()
        d = pd.DataFrame({"study_id": d.uid, "txt": (d.findings.map(clean) + " " + d.impression.map(clean)).str.strip()})
    else:
        d = pd.read_csv(a.reports)[[a.id_col, a.text_col]].rename(columns={a.id_col: "study_id", a.text_col: "txt"})
    d = d[d.txt.fillna("").str.split().str.len() >= 5].sample(frac=1, random_state=a.seed).head(a.n)
    rng = np.random.default_rng(a.seed)
    kinds = ["omission", "insertion", "side confusion", "spelling error", "other", "hedge"]
    rows = []
    for r in d.itertuples():
        for k in kinds:
            e, desc = inject(r.txt, k, rng)
            if e and e.strip() != r.txt.strip():
                rows.append(dict(study_id=r.study_id, original_report=r.txt, error_report=e, error_type=k,
                                 error_description=desc))
    out = pd.DataFrame(rows)
    p = os.path.join(a.out, "injected_errors.csv")
    out.to_csv(p, index=False)
    print(f"E3 wrote {p}: {len(out)} pairs from {len(d)} reports; per type {out.error_type.value_counts().to_dict()}")
    return p


if __name__ == "__main__":
    main()

In [ ]:
import inject_errors as IJ, corbenchx_run as CB
def _inject():
    if IU_DATA_DIR and os.path.exists(IU_DATA_DIR):
        src, args, name = "IU/Open-I", ["--reports", IU_DATA_DIR, "--iu"], "IU"
    elif os.path.exists(os.path.join(RD_OUT, "rexgradient_reports.csv")):
        src, args, name = "ReXGradient", ["--reports", os.path.join(RD_OUT, "rexgradient_reports.csv")], "ReXGradient"
    else:
        raise SystemExit("needs IU_DATA_DIR, or E1 to have run")
    p = IJ.main(args + ["--n", "1000" if QUICK else "5000", "--out", RD_OUT])
    return CB.main(["--corbenchx-path", p, "--prefix", "injected", "--title", f"E3 Injected errors on {name} reports", "--out", RD_OUT])
_src = IU_DATA_DIR or (os.path.join(RD_OUT, "rexgradient_reports.csv") if os.path.exists(os.path.join(RD_OUT, "rexgradient_reports.csv")) else None)
run_d("E3", "Injected errors", _src, _inject)
show_md("injected_summary.md")

## E4. Status of every real-data step (Parts D and E)

In [ ]:
rows = [dict(step=k, dataset=v[0], status=v[1]) for k, v in sorted(D_STATUS.items())]
st = pd.DataFrame(rows)
display(st)
with open(os.path.join(RD_OUT, "table_vii_status.md"), "w") as f:
    f.write("| step | dataset | status |\n|---|---|---|\n" + "".join(f"| {r['step']} | {r['dataset']} | {r['status']} |\n" for r in rows))
print("Results that ran are in", RD_OUT, "(*_summary.md). Copy them into Sec. III-C and replace the corresponding \\tbd placeholders.")